# Solaris RL -- Kaggle Training Notebook

Resumable DQN / Double DQN training for Atari Solaris.

**Kaggle session limit: 9 hours, hard cutoff.** This notebook is designed to be
re-run across multiple sessions -- it auto-resumes from the last saved checkpoint
each time, using Kaggle's persistent **Output** directory to carry checkpoints
between sessions.

### How to use this notebook across multiple sessions
1. **First run:** Settings (right sidebar) -> Accelerator -> **GPU T4 x2**. Run all cells.
2. When the 9-hour limit hits (or you stop it manually), the notebook stops --
 but checkpoints are already saved in `/kaggle/working/checkpoints/`.
3. **To continue:** open this same notebook again (it auto-saves your edits),
 click **Save Version -> Save & Run All**. Kaggle restores your previous
 `/kaggle/working/` output automatically, so checkpoints persist and
 training resumes exactly where it left off.
4. Repeat until `total_steps` is reached.

### Before you start
- Push your `atari-solaris` project to a **public (or Kaggle-accessible private) GitHub repo** first.
- Set `REPO_URL`, `BRANCH`, `CONFIG_PATH`, and `SEED` in Cell 3.
- Metrics are written to TensorBoard under `checkpoints/<exp_name>/tb_logs/`.
 Download that folder after the run and view with `tensorboard --logdir tb_logs`.

### Running a different experiment (e.g. dqn_baseline vs double_dqn)
1. Open the notebook, edit Cell 3:
 - `CONFIG_PATH = "configs/dqn_baseline.yaml"` (or whichever experiment you want)
 - Optionally change `SEED` for a different random seed
2. Click **Save Version -> Save & Run All**. The clone cell detects existing
 checkpoints and resumes; the new experiment's checkpoints land in their own
 folder (`checkpoints/<exp_name>/`), so the two experiments coexist on Output.

### Run playbook (read this once, then ignore)

This notebook has 13 cells in this order:
 1. Intro (markdown, this section)
 2. Playbook (markdown, you are here)
 3. Warnings filter + keep-alive thread
 4. CONFIG (edit REPO_URL, BRANCH, CONFIG_PATH, SEED)
 5. Clone repo with stash/restore
 6. Install dependencies
 7. Verify GPU + env
 8. Smoketest (~3 min, validates whole pipeline)
 9. TensorBoard log dir setup
 10. Symlink assert + show existing checkpoints
 11. Train (the long one, ~4-5 hours for 3M steps)
 12. Record videos (trained + random, ~4 min)
 13. Package results and trigger download
 14. Footer (markdown)

### Normal flow

1. First run on Kaggle: **Save Version -> Save & Run All** (entire notebook).
2. After 9h disconnect (or manual stop): open the same notebook, click **Save Version -> Save & Run All** again. Everything resumes automatically.
3. When you see Cell 13's FileLink, click it to download `results.zip`. Done.

### Per-cell "what to run next"

| Cell | What it does | What to run next |
|------|--------------|------------------|
| 4 | CONFIG: edit REPO_URL/CONFIG_PATH/SEED | Cell 5 (clone) |
| 5 | Clone repo (idempotent, stashes checkpoints) | Cell 6 (install) |
| 6 | Install pip deps | Cell 7 (verify env) |
| 7 | Verify CUDA + gym.make works | Cell 8 (smoketest) |
| 8 | Smoketest 5K steps, ~3 min, asserts pass | Cell 11 (train) |
| 9 | Make checkpoints/ dir | Cell 10 (symlink) |
| 10 | Assert ckpts under /kaggle/working/, list existing | Cell 11 (train) |
| 11 | Train 3M steps, ~4-5h. May time out, that's fine | Cell 12 (videos) |
| 12 | Record videos (~4 min) | Cell 13 (download) |
| 13 | Zip results and trigger FileLink download | Done. Download zip. |

### Per-cell "what if it fails"

| Cell | Failure | Fix |
|------|---------|-----|
| 4 | (nothing should fail) | Just edit the values. Cell only reads yaml. |
| 5 | git clone fails | Check REPO_URL is correct and the repo is public/accessible |
| 5 | git pull fails | The `|| echo 'pull skipped'` handles this; if clone succeeded, proceed |
| 6 | pip install hangs/fails | Re-run the cell; PyPI usually retries fine. If a package is broken, comment it out and re-run |
| 7 | CUDA not available | Sidebar -> Accelerator -> GPU T4 x2 -> Save version -> re-run |
| 7 | gym.make fails | Re-run Cell 6 (install). Some pkg may have failed silently |
| 8 | AssertionError: no checkpoint saved | Check the `tail -30` output above the assert; train.py crashed. Common cause: import error in train.py after a code change |
| 8 | AssertionError: no TensorBoard events | Check `use_tensorboard: true` in your config yaml |
| 11 | `Loss: nan` or Q-values diverge | Rare. Kill session, reduce lr in config (0.0001 -> 0.00005), re-run with --resume |
| 11 | Session times out mid-run | Just re-run the whole notebook. --resume picks up from latest step_*.pt |
| 12 | `exp_name` undefined | Fallback auto-picks most recent experiment. If still fails, run Cell 4 first |
| 12 | Trained video missing | Run was interrupted before `final.pt` was saved. Random baseline still recorded |
| 13 | FileLink doesn't work | Use the Output tab on the right sidebar instead |
| 13 | exp_name fallback fails | Run Cell 4 (CONFIG) first to set exp_name |

### Multi-experiment workflow (e.g. dqn_baseline + double_dqn)

1. First run with `CONFIG_PATH = "configs/double_dqn.yaml"` -> Save & Run All
2. Download results.zip from Cell 13
3. Edit Cell 4: change `CONFIG_PATH = "configs/dqn_baseline.yaml"`
4. Save & Run All again. Cell 5 stashes the existing checkpoints (a no-op since clone is idempotent), Cell 8 smoketest re-runs, Cell 11 starts a fresh experiment in its own folder `checkpoints/dqn_baseline_seed616/`
5. Download the new results.zip. Both runs coexist on Kaggle Output and locally.

### When you just want to resume an interrupted run

1. Open the same notebook
2. **Do not edit Cell 4** -- CONFIG_PATH should stay the same as your previous run
3. Save Version -> Save & Run All
4. Cell 5 stash/restore is a no-op (same checkpoint folder)
5. Cell 11 (train) finds the latest `step_*.pt` and continues

### Clear-output / start-fresh-from-scratch

To abandon a previous run and start fresh:
1. Sidebar -> Save Version -> "Save Output" dropdown -> "Clear outputs and save"
   (This wipes `/kaggle/working/` including checkpoints/.)
2. Edit Cell 4 to your desired CONFIG_PATH and SEED.
3. Save Version -> Save & Run All.

### Local viewing

After downloading `results.zip` and extracting:
```
tensorboard --logdir checkpoints/double_dqn_baseline_seed616/tb_logs
```
Then open `http://localhost:6006` in your browser.

In [ ]:
# ============================================================
# 1. Warnings + Keep-alive thread
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Kaggle disconnects if the browser tab is idle too long (websocket
# timeout). This snippet keeps the session alive by pinging the page
# every 60 seconds. It runs in a background thread; the cell still
# finishes immediately, so subsequent cells are not blocked.
import threading, time
try:
	from IPython.display import Javascript, display

	def _keepalive():
		while True:
			time.sleep(60)
			try:
				display(Javascript("document.querySelector('body').click()"), update=False)
			except Exception:
				pass

	threading.Thread(target=_keepalive, daemon=True).start()
	print("Keep-alive thread started (pings every 60s)")
except ImportError:
	print("Keep-alive skipped (not on Jupyter/IPython)")

In [ ]:
# ============================================================
# CONFIG -- edit these before running
# ============================================================
REPO_URL = "https://github.com/3rdDerivativeofPi/atari-solaris.git" # <-- EDIT THIS
BRANCH = "main"

# Which experiment config to run. Must match a file in configs/.
CONFIG_PATH = "configs/double_dqn.yaml"
SEED = 616

# Derive exp_name from the config (matches what train.py will use).
# exp_name = "{experiment.name}_seed{SEED}"
# It's used by later cells for checkpoint paths and video output.
import yaml
_cfg = yaml.safe_load(open(CONFIG_PATH, encoding='utf-8'))
exp_name = f"{_cfg['experiment']['name']}_seed{SEED}"
del _cfg
print(f"exp_name = {exp_name}")

In [ ]:
# ============================================================
# 2. Clone (or update) the repo into /kaggle/working
# ============================================================
# /kaggle/working persists across sessions for the SAME notebook, so we
# clone once and `git pull` on subsequent runs.
#
# IMPORTANT: a fresh clone or a `git pull` after a force-push would nuke
# our checkpoints/. Stash any existing checkpoints/ to /kaggle/working/
# _ckpt_stash BEFORE the clone, restore AFTER. This makes the cell
# idempotent -- safe to re-run multiple times across sessions.
import os, shutil

PROJECT_DIR = "/kaggle/working/atari-solaris"
_STASH = "/kaggle/working/_ckpt_stash"

if os.path.isdir(_STASH):
	shutil.rmtree(_STASH)
if os.path.isdir(f"{PROJECT_DIR}/checkpoints"):
	shutil.move(f"{PROJECT_DIR}/checkpoints", _STASH)
if os.path.isdir(PROJECT_DIR):
	shutil.rmtree(PROJECT_DIR)

os.chdir("/kaggle/working")
!git clone --branch {BRANCH} {REPO_URL}

os.chdir(PROJECT_DIR)
!git pull origin {BRANCH} || echo 'pull skipped (offline?)'

# Restore stashed checkpoints/ (if any). The fresh repo has no
# checkpoints/ directory, so this just merges the saved state back in.
if os.path.isdir(_STASH):
	if os.path.isdir(f"{PROJECT_DIR}/checkpoints"):
		shutil.rmtree(f"{PROJECT_DIR}/checkpoints")
	shutil.move(_STASH, f"{PROJECT_DIR}/checkpoints")
	print(f"[resume] restored checkpoint directory from {_STASH}")
else:
	os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)

print("\nWorking directory:", os.getcwd())

In [ ]:
# ============================================================
# 2. Install dependencies
# ============================================================
# Kaggle images already ship PyTorch + CUDA, so we skip reinstalling torch
# (reinstalling it tends to break the preconfigured CUDA linkage).
!pip install -q ale-py gymnasium[accept-rom-license] AutoROM opencv-python-headless tensorboard pyyaml imageio imageio-ffmpeg
!AutoROM --accept-license -y > /dev/null 2>&1
print("OK Dependencies installed")

In [ ]:
# ============================================================
# 3. Verify GPU and environment
# ============================================================
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
	print("GPU:", torch.cuda.get_device_name(0))

import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
env = gym.make("ALE/Solaris-v5", frameskip=1)
obs, _ = env.reset()
print("[OK] Solaris env OK -- obs shape:", obs.shape)
env.close()

In [ ]:
# ============================================================
# 3b. Smoketest (~3 min on T4)
# ============================================================
# Run a tiny 5,000-step training before committing to the multi-hour run.
# Catches typos, import errors, env misconfigurations, etc. early.
# Asserts: checkpoint file exists, tb_logs written.
#
# Set SMOKETEST = False to skip once the smoketest has passed in a prior
# session -- just running cell 10 (training) is enough.
#
# After this cell: re-run it with SMOKETEST=False, or just go to Cell 10.

In [ ]:
# ============================================================
# 4. Set up TensorBoard log directory (local-only metrics)
# ============================================================
# Metrics are written to /kaggle/working/checkpoints/<exp_name>/tb_logs/
# which is part of Kaggle's persistent Output. Download that folder after
# the run and view locally with:
# tensorboard --logdir tb_logs
import os
TB_ROOT = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(TB_ROOT, exist_ok=True)
print("TensorBoard logs will be written under:", TB_ROOT)

In [ ]:
# ============================================================
# 5. Symlink checkpoints into /kaggle/working so they PERSIST
#    across sessions (this is the critical resumability step)
# ============================================================
# Everything under /kaggle/working is saved as this notebook's "Output" and
# restored automatically the next time you open and run this same notebook.
# checkpoints/ already lives inside PROJECT_DIR, which is itself inside
# /kaggle/working, so no extra symlinking is actually needed -- but we assert
# it here explicitly so it's obvious and verifiable.
ckpt_root = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(ckpt_root, exist_ok=True)
assert ckpt_root.startswith("/kaggle/working"), "Checkpoints must live under /kaggle/working to persist!"
print("[OK] Checkpoints will persist at:", ckpt_root)

# Show what's already there (non-empty on a resumed session)
for root, dirs, files in os.walk(ckpt_root):
	for f in files:
		print(" -", os.path.join(root, f))

In [ ]:
# ============================================================
# 6. Train -- auto-resumes if a checkpoint already exists
# ============================================================
# --resume is always passed:
# - No checkpoint found -> train.py starts fresh
# - Checkpoint found -> weights/optimiser/step restored
#
# Safety nets against Kaggle disconnects:
# - python -u  : unbuffered stdout/stderr, so logs stream live
# - 2>&1 | tee train.log : also persist a copy under checkpoints/
#   so even if the cell output is truncated, the logfile survives
#   in Kaggle Output and can be downloaded alongside checkpoints.
LOG_PATH = os.path.join(PROJECT_DIR, "checkpoints", "train.log")
!python -u train.py --config {CONFIG_PATH} --seed {SEED} --resume 2>&1 | tee {LOG_PATH}

In [ ]:
# ============================================================
# Record videos (trained agent + random baseline)
# ============================================================
# Records MP4s side-by-side for visual comparison. Bypasses
# gymnasium's RecordVideo (fps=None crash on Kaggle) and uses
# imageio directly.
#
# Videos live under videos/ and are included in the results.zip
# from the next cell. Set RECORD_VIDEO = False to skip if running
# short on time (each recording takes ~2 min per episode).
#
# After this cell: go to the next cell (package and download).
import os
os.chdir(PROJECT_DIR)

# exp_name is set in the CONFIG cell. If you ran cells out of order,
# fall back to inferring it from the experiments on disk.
if 'exp_name' not in dir():
	ckpts_root = os.path.join(PROJECT_DIR, "checkpoints")
	if os.path.isdir(ckpts_root):
		exps = [d for d in os.listdir(ckpts_root)
			if os.path.isdir(os.path.join(ckpts_root, d)) and not d.startswith('_')]
		if exps:
			exps.sort(key=lambda d: os.path.getmtime(os.path.join(ckpts_root, d)),
				reverse=True)
			exp_name = exps[0]
			print(f"[warn] exp_name not in scope; using most recent: {exp_name}")
		else:
			raise RuntimeError("exp_name undefined and no checkpoints/ experiments found")

RECORD_VIDEO = True
if RECORD_VIDEO:
	# Clean any prior videos so the zip is fresh
	vdir = os.path.join(PROJECT_DIR, "videos")
	if os.path.isdir(vdir):
		for n in os.listdir(vdir):
			full = os.path.join(vdir, n)
			if os.path.isfile(full):
				os.remove(full)

	ckpt_path = os.path.join(PROJECT_DIR, "checkpoints", exp_name, "final.pt")

	if os.path.exists(ckpt_path):
		# Trained agent
		!python scripts/record_video.py --checkpoint {ckpt_path} --episodes 1 --output videos/trained.mp4
	else:
		print(f"[skip] no checkpoint at {ckpt_path}; skipping trained video")

	# Random baseline (always, for comparison)
	!python scripts/record_video.py --episodes 1 --output videos/random.mp4

	print("\n[videos] written to:", vdir)
	for n in sorted(os.listdir(vdir)):
		full = os.path.join(vdir, n)
		print(f" {n} ({os.path.getsize(full):,} bytes)")
else:
	print("[videos] skipped")

In [ ]:
# ============================================================
# Package results and trigger download
# ============================================================
# After the run finishes, this cell:
# 1. Keeps only the most recent N checkpoints (final.pt + last 3 step_*.pt)
# 2. Zips the experiment folder + videos into /kaggle/working/results.zip
# 3. Creates a FileLink -- click it to download
#
# If the FileLink times out or errors on a large archive, the zip
# also lives in Kaggle's persistent Output directory (top-right
# "Save Version" -> "Output" tab), so you can always fetch it from
# there too.
#
# After this cell: download results.zip from the link below, or from
# the Output tab on the right sidebar.
import os, glob, shutil
from IPython.display import FileLink

# Fallback for exp_name if cells were run out of order
if 'exp_name' not in dir():
	ckpts_root = os.path.join(PROJECT_DIR, "checkpoints")
	exps = sorted([d for d in os.listdir(ckpts_root)
		if os.path.isdir(os.path.join(ckpts_root, d)) and not d.startswith('_')],
		key=lambda d: os.path.getmtime(os.path.join(ckpts_root, d)),
		reverse=True)
	if exps:
		exp_name = exps[0]
		print(f"[warn] exp_name not in scope; using most recent: {exp_name}")
	else:
		raise RuntimeError("exp_name undefined and no checkpoints/ experiments found")

KEEP_LAST_N_CKPTS = 3 # keep final.pt + last N step_*.pt
exp_dir = os.path.join(PROJECT_DIR, "checkpoints", exp_name)

# 1. Prune old checkpoints (keep only the latest N + final.pt)
ckpts = sorted(glob.glob(os.path.join(exp_dir, "step_*.pt")))
for old_ckpt in ckpts[:-KEEP_LAST_N_CKPTS]:
	os.remove(old_ckpt)
print(f"Kept last {min(KEEP_LAST_N_CKPTS, len(ckpts))} checkpoints + final.pt")

# 2. Zip the experiment folder (checkpoints + tb_logs + train.log)
# plus the videos/ folder if it exists (recorded in the previous cell)
ZIP_PATH = "/kaggle/working/results.zip"
if os.path.exists(ZIP_PATH):
	os.remove(ZIP_PATH)

# First zip: the experiment folder
exp_zip = "/kaggle/working/_exp_tmp"
if os.path.exists(exp_zip):
	shutil.rmtree(exp_zip)
shutil.make_archive(exp_zip, "zip",
	root_dir=PROJECT_DIR,
	base_dir=os.path.join("checkpoints", exp_name))
shutil.move(exp_zip + ".zip", ZIP_PATH)

# Second zip: append videos/ if present
videos_dir = os.path.join(PROJECT_DIR, "videos")
if os.path.isdir(videos_dir) and os.listdir(videos_dir):
	import zipfile
	with zipfile.ZipFile(ZIP_PATH, "a") as zf:
		for n in os.listdir(videos_dir):
			full = os.path.join(videos_dir, n)
			if os.path.isfile(full):
				zf.write(full, arcname=os.path.join("videos", n))
	print(f"Added {len(os.listdir(videos_dir))} video(s) to archive")

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"Archive created: {ZIP_PATH} ({size_mb:.1f} MB)")

# 3. Trigger download
FileLink(ZIP_PATH, result_html_prefix=f"Click to download ({size_mb:.1f} MB): ")

---
### What happens when this session times out mid-training

Nothing bad -- that's the point. The training loop checkpoints every
`checkpoint_freq` steps (25K by default, ~2 min on T4) to
`checkpoints/<exp_name>/`. When you come back and re-run this notebook:

- The repo is already cloned (just `git pull`s for any code updates).
- `train.py --resume` finds the latest `step_*.pt` checkpoint and restores
 network weights, optimiser state, and the step counter.
- The replay buffer starts empty and refills over the first `learning_starts`
 steps -- a small, expected cost, not a bug.
- TensorBoard logs under `checkpoints/<exp_name>/tb_logs/` accumulate
 across sessions, so you get one continuous learning curve.

### Estimating sessions needed
At ~150-300 SPS on a Kaggle T4 (variable with load), a 3M-step run takes
roughly **4-5 hours of GPU time total** -> **1 Kaggle session** at the
9-hour cap (with time to spare for smoketest, video recording, and zip).
Kaggle gives 30 free GPU-hours/week, which covers this easily.